<a href="https://colab.research.google.com/github/Varshu006/Stock-Price-Predictor/blob/main/Stock_Prediction_ML.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q yfinance xgboost gradio pandas numpy scikit-learn plotly

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import gradio as gr

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score

# ==============================================================================
# 1. CORE ML PIPELINE FUNCTION
# ==============================================================================
def train_and_predict(ticker, start_date, end_date):
    try:
        # A. Ingest Data via yfinance
        df = yf.download(ticker.upper(), start=start_date, end=end_date)

        if df.empty:
            return "Error: No data found for ticker.", None, None

        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df = df[['Open', 'High', 'Low', 'Close', 'Volume']].dropna()

        # B. Feature Engineering
        df['SMA_10'] = df['Close'].rolling(window=10).mean()
        df['SMA_50'] = df['Close'].rolling(window=50).mean()

        # RSI (14 days)
        delta = df['Close'].diff()
        gain = (delta.where(delta > 0, 0)).rolling(window=14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(window=14).mean()
        rs = gain / loss
        df['RSI_14'] = 100 - (100 / (1 + rs))

        # Volatility
        df['Daily_Return'] = df['Close'].pct_change()
        df['Volatility_10'] = df['Daily_Return'].rolling(window=10).std()

        # Target: 1 if tomorrow's Close > today's Close, else 0
        df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)
        df.dropna(inplace=True)

        # C. Time-Series Train/Test Split (80/20)
        features = ['Open', 'High', 'Low', 'Close', 'Volume', 'SMA_10', 'SMA_50', 'RSI_14', 'Volatility_10']
        X = df[features]
        y = df['Target']

        split_idx = int(len(df) * 0.8)
        X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
        y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

        # D. Train Model
        model = XGBClassifier(
            n_estimators=100,
            learning_rate=0.03,
            max_depth=3,
            random_state=42
        )
        model.fit(X_train, y_train)

        # E. Model Performance & Latest Signal
        y_pred = model.predict(X_test)
        accuracy = accuracy_score(y_test, y_pred)

        # Predict for tomorrow using latest available row
        latest_features = X.iloc[[-1]]
        tomorrow_pred = model.predict(latest_features)[0]
        prob = model.predict_proba(latest_features)[0]

        signal = "🟢 BULLISH (BUY)" if tomorrow_pred == 1 else "🔴 BEARISH (SELL / CASH)"
        confidence = prob[1] if tomorrow_pred == 1 else prob[0]

        summary_text = (
            f"### Stock Analysis: {ticker.upper()}\n"
            f"- **Next Day Prediction:** {signal}\n"
            f"- **Model Confidence:** {confidence:.2%}\n"
            f"- **Test Set Directional Accuracy:** {accuracy:.2%}\n"
            f"- **Latest Close Price:** ${df['Close'].iloc[-1]:.2f}\n"
            f"- **RSI (14):** {df['RSI_14'].iloc[-1]:.2f}\n"
        )

        # F. Plotly Interactive Chart
        fig = go.Figure()

        # Historical Close
        fig.add_trace(go.Scatter(
            x=df.index[split_idx:], y=df['Close'].iloc[split_idx:],
            mode='lines', name='Actual Price', line=dict(color='gray', width=1)
        ))

        # Moving Averages
        fig.add_trace(go.Scatter(
            x=df.index[split_idx:], y=df['SMA_10'].iloc[split_idx:],
            mode='lines', name='SMA 10', line=dict(color='blue', width=1.5)
        ))

        fig.add_trace(go.Scatter(
            x=df.index[split_idx:], y=df['SMA_50'].iloc[split_idx:],
            mode='lines', name='SMA 50', line=dict(color='orange', width=1.5)
        ))

        fig.update_layout(
            title=f"{ticker.upper()} Test Period Price & Technical Indicators",
            xaxis_title="Date",
            yaxis_title="Price ($)",
            template="plotly_white",
            height=500
        )

        return summary_text, fig

    except Exception as e:
        return f"Execution Error: {str(e)}", None

# ==============================================================================
# 2. GRADIO WEB USER INTERFACE
# ==============================================================================
with gr.Blocks(title="ML Stock Predictor Prototype") as demo:
    gr.Markdown("# 📈 Machine Learning Stock Price Predictor")
    gr.Markdown("Interactive prototype fetching live data via **yfinance** and making directional trend predictions using **XGBoost**.")

    with gr.Row():
        with gr.Column(scale=1):
            ticker_input = gr.Textbox(value="AAPL", label="Stock Ticker Symbol")
            start_date = gr.Textbox(value="2020-01-01", label="Start Date (YYYY-MM-DD)")
            end_date = gr.Textbox(value="2024-01-01", label="End Date (YYYY-MM-DD)")
            btn = gr.Button("Run ML Prediction", variant="primary")

        with gr.Column(scale=2):
            output_text = gr.Markdown(label="Prediction Results")
            output_plot = gr.Plot(label="Price & Indicators Chart")

    btn.click(
        fn=train_and_predict,
        inputs=[ticker_input, start_date, end_date],
        outputs=[output_text, output_plot]
    )

# Launch with public link in Google Colab
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://83dab73c94b567da0a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


/tmp/ipykernel_9948/1644492589.py:16: FutureWarning: YF.download() has changed argument auto_adjust default to True
  df = yf.download(ticker.upper(), start=start_date, end=end_date)
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_9948/1644492589.py:16: FutureWarning:

YF.download() has changed argument auto_adjust default to True

[*********************100%***********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['AAPL']: YFPricesMissingError('possibly delisted; no price data found  (1d 2026-09-20 -> 2026-09-22)')
/usr/local/lib/python3.13/dist-packages/gradio/blocks.py:1971: UserWarning:

A function (train_and_predict) returned too many output values (needed: 2, returned: 3). Ignoring extra values.
    Output components:
        [markdown, plot]
    Output values returned:
        ["Error: No data found for ticker.", None, None]

/tmp/ipykernel_9948/1644492589.py:16: FutureWarning:

YF.download() has changed argu